In [ ]:
"""
State-of-the-Art Baseline: Stacked Bidirectional LSTM with Attention

Task      : Single-task regression — 4 independent models
Targets   : PM2.5, NO2, CO, Ozone
Metrics   : RMSE, MAE, R²
Baseline  : For comparison against a Multi-Task Learning (MTL) model

"""

import os
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns

warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)


TRAIN_PATH   = "Train_data.csv"
VAL_PATH     = "Validation_data.csv"
TEST_PATH    = "Test_data.csv"
OUTPUT_DIR   = "sota_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOOKBACK     = 24      
LSTM_UNITS   = 128      
DROPOUT_RATE = 0.4      
DENSE_UNITS  = 64
ATTN_UNITS   = 64
BATCH_SIZE   = 64       
MAX_EPOCHS   = 50       
LEARNING_RATE = 0.001   


TARGETS = {
    "PM2.5": "pm25",
    "NO2":   "no2",
    "CO":    "co",
    "Ozone": "ozone",
}


POLLUTANT_FEATURES = ["pm10", "no", "nh3", "nox", "so2"]
METEO_FEATURES     = ["bp", "wind_speed", "air_temp", "humidity", "rainfall"]
CYCLICAL_FEATURES  = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
ALL_FEATURES       = POLLUTANT_FEATURES + METEO_FEATURES + CYCLICAL_FEATURES





def load_dataframe(path: str) -> pd.DataFrame:
    """Load CSV, parse datetime, sort chronologically, add cyclical features."""
    df = pd.read_csv(path, parse_dates=["from_date"])
    df = df.sort_values("from_date").reset_index(drop=True)

    # Cyclical time features (paper §IV.B)
    df["hour"]     = df["from_date"].dt.hour
    df["dow"]      = df["from_date"].dt.dayofweek
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"]  = np.sin(2 * np.pi * df["dow"]  / 7)
    df["dow_cos"]  = np.cos(2 * np.pi * df["dow"]  / 7)

    
    df = df.fillna(method="ffill").fillna(method="bfill")
    return df


def build_arrays_for_target(
    train_df: pd.DataFrame,
    val_df:   pd.DataFrame,
    test_df:  pd.DataFrame,
    target_col: str,
):
    """
    For a given target pollutant:
      - Input features  = ALL_FEATURES + the other 3 target pollutants
      - Target          = target_col (raw, unscaled for inverse transform later)

    Returns scaled X arrays, raw y arrays, feature scaler, target scaler.
    """
    other_targets = [col for col in TARGETS.values() if col != target_col]
    feature_cols  = ALL_FEATURES + other_targets          

    
    train_df = train_df.dropna(subset=[target_col])
    val_df   = val_df.dropna(subset=[target_col])
    test_df  = test_df.dropna(subset=[target_col])

    X_tr_raw = train_df[feature_cols].values.astype(np.float32)
    X_va_raw = val_df[feature_cols].values.astype(np.float32)
    X_te_raw = test_df[feature_cols].values.astype(np.float32)

    y_tr = train_df[target_col].values.astype(np.float32)
    y_va = val_df[target_col].values.astype(np.float32)
    y_te = test_df[target_col].values.astype(np.float32)

    
    feat_scaler = RobustScaler()
    X_tr = feat_scaler.fit_transform(X_tr_raw)
    X_va = feat_scaler.transform(X_va_raw)
    X_te = feat_scaler.transform(X_te_raw)

    
    tgt_scaler = RobustScaler()
    y_tr_scaled = tgt_scaler.fit_transform(y_tr.reshape(-1, 1)).ravel()
    y_va_scaled = tgt_scaler.transform(y_va.reshape(-1, 1)).ravel()
    # y_te stays raw — we inverse-transform predictions for evaluation

    return (X_tr, y_tr_scaled,
            X_va, y_va_scaled,
            X_te, y_te,
            feat_scaler, tgt_scaler,
            feature_cols)


def make_sequences(X: np.ndarray, y: np.ndarray, lookback: int = LOOKBACK):
    """Sliding window → (N, lookback, n_features), (N,)"""
    Xs, ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback: i])
        ys.append(y[i])
    return np.array(Xs, dtype=np.float32), np.array(ys, dtype=np.float32)



class BahdanauAttention(layers.Layer):
    """
    Additive attention over Bi-LSTM hidden states.

        e_t = v^T · tanh(W_h · h_t + b_a)
        α_t = softmax(e)
        c   = Σ α_t · h_t
    """
    def __init__(self, units: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, use_bias=False)
        self.V = layers.Dense(1,     use_bias=False)

    def call(self, hidden_states):
        score  = self.V(tf.nn.tanh(self.W(hidden_states)))   # (B, T, 1)
        alpha  = tf.nn.softmax(score, axis=1)                 # (B, T, 1)
        context = tf.reduce_sum(alpha * hidden_states, axis=1) # (B, 2*units)
        return context, tf.squeeze(alpha, -1)

    def get_config(self):
        return super().get_config()




def build_model(
    timesteps:    int,
    n_features:   int,
    lstm_units:   int   = LSTM_UNITS,
    dropout_rate: float = DROPOUT_RATE,
    dense_units:  int   = DENSE_UNITS,
    attn_units:   int   = ATTN_UNITS,
    lr:           float = LEARNING_RATE,
    target_name:  str   = "pollutant",
) -> Model:
    """
    Stacked Bi-LSTM (2×128) + Bahdanau Attention + Dense regression head.

    Input  → (None, 24, n_features)
    BiLSTM₁ return_sequences=True  → (None, 24, 256)
    Dropout 0.4
    BiLSTM₂ return_sequences=True  → (None, 24, 256)
    Dropout 0.4
    BahdanauAttention               → (None, 256)
    Dense 64 ReLU
    Dense 1  Linear                 → predicted pollutant value (scaled)
    """
    inp = keras.Input(shape=(timesteps, n_features), name="input")

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True), name="bilstm_1"
    )(inp)
    x = layers.Dropout(dropout_rate, name="dropout_1")(x)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True), name="bilstm_2"
    )(x)
    x = layers.Dropout(dropout_rate, name="dropout_2")(x)

    context, _ = BahdanauAttention(units=attn_units, name="attention")(x)

    x   = layers.Dense(dense_units, activation="relu", name="dense_1")(context)
    out = layers.Dense(1, activation="linear", name="output")(x)

    model = Model(inputs=inp, outputs=out,
                  name=f"BiLSTM_Attn_{target_name}")

    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=lr),
        loss="mse",
        metrics=["mae"],
    )
    return model




def train_single_model(model, X_tr, y_tr, X_va, y_va, target_name):
    ckpt_path = os.path.join(OUTPUT_DIR, f"best_{target_name}.keras")
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=5,
            restore_best_weights=True, verbose=0
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5, patience=3, verbose=0
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=ckpt_path, monitor="val_loss",
            save_best_only=True, verbose=0
        ),
    ]

    history = model.fit(
        X_tr, y_tr,
        validation_data=(X_va, y_va),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        shuffle=False,           
        verbose=1,
    )
    return history




def evaluate_model(model, X_te, y_te_raw, tgt_scaler, target_name):
    """Predict on test set, inverse-transform, compute RMSE / MAE / R²."""
    y_pred_scaled = model.predict(X_te, verbose=0).ravel()
    y_pred = tgt_scaler.inverse_transform(
        y_pred_scaled.reshape(-1, 1)
    ).ravel()

    rmse = np.sqrt(mean_squared_error(y_te_raw, y_pred))
    mae  = mean_absolute_error(y_te_raw, y_pred)
    r2   = r2_score(y_te_raw, y_pred)

    print(f"  {target_name:<8}  RMSE={rmse:.4f}  MAE={mae:.4f}  R²={r2:.4f}")
    return {"target": target_name, "RMSE": rmse, "MAE": mae, "R2": r2,
            "y_true": y_te_raw, "y_pred": y_pred}



def plot_results(all_results: list, all_histories: dict):
    """
    4-panel figure per pollutant: learning curve + prediction vs actual.
    Plus a final summary bar chart comparing RMSE / MAE / R² across pollutants.
    """
    palette = {"PM2.5": "#2563eb", "NO2": "#16a34a",
               "CO":    "#dc2626", "Ozone": "#ca8a04"}

    # ── Per-pollutant plots ───────────────────────────────────────────────
    for res in all_results:
        name    = res["target"]
        hist    = all_histories[name]
        color   = palette[name]
        y_true  = res["y_true"]
        y_pred  = res["y_pred"]

        fig, axes = plt.subplots(1, 3, figsize=(18, 4))
        fig.suptitle(
            f"SOTA Bi-LSTM+Attention — {name} (Delhi)\n"
            f"RMSE={res['RMSE']:.3f}  MAE={res['MAE']:.3f}  R²={res['R2']:.3f}",
            fontsize=12, fontweight="bold"
        )

        # Loss curve
        ax = axes[0]
        ax.plot(hist.history["loss"],     color=color,    label="Train Loss")
        ax.plot(hist.history["val_loss"], color=color,    label="Val Loss",
                linestyle="--", alpha=0.6)
        ax.set_title("Loss (MSE)"); ax.set_xlabel("Epoch")
        ax.legend(); ax.grid(alpha=0.3)

       
        ax = axes[1]
        n = min(500, len(y_true))
        ax.plot(y_true[:n], label="Actual",    color="black", linewidth=0.8)
        ax.plot(y_pred[:n], label="Predicted", color=color,
                linewidth=0.8, linestyle="--", alpha=0.85)
        ax.set_title(f"Prediction vs Actual (first {n} steps)")
        ax.set_xlabel("Time Step"); ax.set_ylabel(f"{name} (µg/m³ or ppm)")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        # Scatter: predicted vs actual
        ax = axes[2]
        ax.scatter(y_true, y_pred, alpha=0.15, s=6, color=color)
        lims = [min(y_true.min(), y_pred.min()),
                max(y_true.max(), y_pred.max())]
        ax.plot(lims, lims, "k--", linewidth=1, label="Perfect fit")
        ax.set_title("Scatter: Actual vs Predicted")
        ax.set_xlabel("Actual"); ax.set_ylabel("Predicted")
        ax.legend(fontsize=8); ax.grid(alpha=0.3)

        plt.tight_layout()
        save_path = os.path.join(OUTPUT_DIR, f"sota_{name.lower()}.png")
        plt.savefig(save_path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"   Plot saved → {save_path}")

    
    names  = [r["target"] for r in all_results]
    rmses  = [r["RMSE"]   for r in all_results]
    maes   = [r["MAE"]    for r in all_results]
    r2s    = [r["R2"]     for r in all_results]
    colors = [palette[n]  for n in names]

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    fig.suptitle("SOTA Single-Task Bi-LSTM+Attention — Summary (Delhi)",
                 fontsize=13, fontweight="bold")

    for ax, values, title, ylabel in zip(
        axes,
        [rmses, maes, r2s],
        ["RMSE (lower is better)", "MAE (lower is better)", "R² (higher is better)"],
        ["RMSE", "MAE", "R²"],
    ):
        bars = ax.bar(names, values, color=colors, edgecolor="white", linewidth=0.5)
        ax.set_title(title); ax.set_ylabel(ylabel)
        ax.grid(axis="y", alpha=0.3)
        for bar, v in zip(bars, values):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(values)*0.01,
                    f"{v:.3f}", ha="center", fontsize=9, fontweight="bold")

    plt.tight_layout()
    summary_path = os.path.join(OUTPUT_DIR, "sota_summary.png")
    plt.savefig(summary_path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   Summary plot saved → {summary_path}")




def main():
    print("─"*60)
    print("  SOTA: 4× Single-Task Stacked Bi-LSTM + Attention (Delhi)")
    print("  Targets: PM2.5 | NO2 | CO | Ozone")
    print("─"*60)


    print("\n[1/3] Loading data …")
    train_df = load_dataframe(TRAIN_PATH)
    val_df   = load_dataframe(VAL_PATH)
    test_df  = load_dataframe(TEST_PATH)
    print(f"   Train: {len(train_df)} rows  |  "
          f"Val: {len(val_df)} rows  |  Test: {len(test_df)} rows")


    print("\n[2/3] Training single-task models …\n")
    all_results   = []
    all_histories = {}

    for display_name, col_name in TARGETS.items():
        print(f"\n{'='*50}")
        print(f"  Target: {display_name}  (column: {col_name})")
        print(f"{'='*50}")

        # Prepare arrays
        (X_tr, y_tr,
         X_va, y_va,
         X_te, y_te_raw,
         feat_scaler, tgt_scaler,
         feature_cols) = build_arrays_for_target(
            train_df, val_df, test_df, col_name
        )

        # Sequences
        X_tr_seq, y_tr_seq = make_sequences(X_tr, y_tr)
        X_va_seq, y_va_seq = make_sequences(X_va, y_va)
        X_te_seq, y_te_seq = make_sequences(X_te, y_te_raw)

        print(f"  Sequences — train: {X_tr_seq.shape}  "
              f"val: {X_va_seq.shape}  test: {X_te_seq.shape}")
        print(f"  Input features ({len(feature_cols)}): {feature_cols}")

        # Build & train
        model = build_model(
            timesteps=LOOKBACK,
            n_features=X_tr_seq.shape[2],
            target_name=display_name,
        )

        history = train_single_model(
            model, X_tr_seq, y_tr_seq,
            X_va_seq, y_va_seq,
            display_name,
        )
        all_histories[display_name] = history

        # Evaluate
        print(f"\n  Test Results:")
        result = evaluate_model(
            model, X_te_seq, y_te_seq, tgt_scaler, display_name
        )
        all_results.append(result)


    print("\n" + "="*55)
    print("  FINAL SUMMARY — SOTA Single-Task Bi-LSTM+Attention")
    print("="*55)
    print(f"  {'Pollutant':<10} {'RMSE':>10} {'MAE':>10} {'R²':>10}")
    print("  " + "-"*42)
    for r in all_results:
        print(f"  {r['target']:<10} {r['RMSE']:>10.4f} "
              f"{r['MAE']:>10.4f} {r['R2']:>10.4f}")
    print("="*55)


    print("\n[3/3] Saving results …")
    metrics_df = pd.DataFrame([
        {"model": "SOTA_BiLSTM_Attention",
         "target": r["target"],
         "RMSE": round(r["RMSE"], 4),
         "MAE":  round(r["MAE"],  4),
         "R2":   round(r["R2"],   4)}
        for r in all_results
    ])
    csv_path = os.path.join(OUTPUT_DIR, "sota_metrics.csv")
    metrics_df.to_csv(csv_path, index=False)
    print(f"   Metrics saved → {csv_path}")


    plot_results(all_results, all_histories)

    print("\nDone ✓")
    return all_results, metrics_df


if __name__ == "__main__":
    all_results, metrics_df = main()

────────────────────────────────────────────────────────────
  SOTA: 4× Single-Task Stacked Bi-LSTM + Attention (Delhi)
  Targets: PM2.5 | NO2 | CO | Ozone
────────────────────────────────────────────────────────────

[1/3] Loading data …
   Train: 52584 rows  |  Val: 1416 rows  |  Test: 744 rows

[2/3] Training single-task models …


  Target: PM2.5  (column: pm25)
  Sequences — train: (52560, 24, 17)  val: (1392, 24, 17)  test: (720, 24, 17)
  Input features (17): ['pm10', 'no', 'nh3', 'nox', 'so2', 'bp', 'wind_speed', 'air_temp', 'humidity', 'rainfall', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'no2', 'co', 'ozone']
Epoch 1/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 15s 14ms/step - loss: 0.1559 - mae: 0.2243 - val_loss: 0.0756 - val_mae: 0.2053 - learning_rate: 0.0010
Epoch 2/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0817 - mae: 0.1639 - val_loss: 0.0631 - val_mae: 0.1855 - learning_rate: 0.0010
Epoch 3/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0761 - mae: 0.154

In [ ]:

import os
import warnings
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers, Model
from sklearn.preprocessing import RobustScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
tf.random.set_seed(42)
np.random.seed(42)


TRAIN_PATH    = "Train_data.csv"
VAL_PATH      = "Validation_data.csv"
TEST_PATH     = "Test_data.csv"
OUTPUT_DIR    = "mtl_outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

LOOKBACK      = 24       # same as SOTA
LSTM_UNITS    = 128      # same as SOTA
DROPOUT_RATE  = 0.4      # same as SOTA
DENSE_UNITS   = 64       # same as SOTA
ATTN_UNITS    = 64       # same as SOTA
BATCH_SIZE    = 64       # same as SOTA
MAX_EPOCHS    = 50       # same as SOTA
LEARNING_RATE = 0.001    # same as SOTA
PATIENCE      = 5        # same as SOTA

# Target pollutants
TARGETS = {
    "PM2.5": "pm25",
    "NO2":   "no2",
    "CO":    "co",
    "Ozone": "ozone",
}
TARGET_KEYS = list(TARGETS.keys())    # ["PM2.5", "NO2", "CO", "Ozone"]
TARGET_COLS = list(TARGETS.values())  # ["pm25",  "no2",  "co",  "ozone"]

# Input features — same as SOTA
POLLUTANT_FEATURES = ["pm10", "no", "nh3", "nox", "so2"]
METEO_FEATURES     = ["bp", "wind_speed", "air_temp", "humidity", "rainfall"]
CYCLICAL_FEATURES  = ["hour_sin", "hour_cos", "dow_sin", "dow_cos"]
ALL_FEATURES       = POLLUTANT_FEATURES + METEO_FEATURES + CYCLICAL_FEATURES
# All 4 target pollutants also used as inputs (same as SOTA)

PALETTE = {"PM2.5": "#2563eb", "NO2": "#16a34a",
           "CO":    "#dc2626", "Ozone": "#ca8a04"}




def load_dataframe(path: str) -> pd.DataFrame:
    """Load CSV, parse datetime, sort, add cyclical features, ffill."""
    df = pd.read_csv(path, parse_dates=["from_date"])
    df = df.sort_values("from_date").reset_index(drop=True)

    df["hour"]     = df["from_date"].dt.hour
    df["dow"]      = df["from_date"].dt.dayofweek
    df["hour_sin"] = np.sin(2 * np.pi * df["hour"] / 24)
    df["hour_cos"] = np.cos(2 * np.pi * df["hour"] / 24)
    df["dow_sin"]  = np.sin(2 * np.pi * df["dow"]  / 7)
    df["dow_cos"]  = np.cos(2 * np.pi * df["dow"]  / 7)

    df = df.fillna(method="ffill").fillna(method="bfill")
    return df


def build_arrays(train_df, val_df, test_df):
    """
    Build feature matrix X and multi-target matrix Y (N, 4).

    Input features = ALL_FEATURES + all 4 target columns (same as SOTA).
    Each target is scaled independently with its own RobustScaler.
    Test targets kept raw for evaluation (predictions are inverse-transformed).
    """
    feature_cols = ALL_FEATURES + TARGET_COLS

    # Drop rows where ANY target is NaN
    train_df = train_df.dropna(subset=TARGET_COLS)
    val_df   = val_df.dropna(subset=TARGET_COLS)
    test_df  = test_df.dropna(subset=TARGET_COLS)

    X_tr_raw = train_df[feature_cols].values.astype(np.float32)
    X_va_raw = val_df[feature_cols].values.astype(np.float32)
    X_te_raw = test_df[feature_cols].values.astype(np.float32)

    # RobustScaler on features (same as SOTA)
    feat_scaler = RobustScaler()
    X_tr = feat_scaler.fit_transform(X_tr_raw)
    X_va = feat_scaler.transform(X_va_raw)
    X_te = feat_scaler.transform(X_te_raw)

    # Scale each target independently
    tgt_scalers  = {}
    Y_tr_scaled  = np.zeros((len(train_df), 4), dtype=np.float32)
    Y_va_scaled  = np.zeros((len(val_df),   4), dtype=np.float32)
    Y_te_raw_arr = np.zeros((len(test_df),  4), dtype=np.float32)

    for i, col in enumerate(TARGET_COLS):
        sc = RobustScaler()
        Y_tr_scaled[:, i]  = sc.fit_transform(
            train_df[col].values.reshape(-1, 1)
        ).ravel()
        Y_va_scaled[:, i]  = sc.transform(
            val_df[col].values.reshape(-1, 1)
        ).ravel()
        Y_te_raw_arr[:, i] = test_df[col].values   
        tgt_scalers[col]   = sc

    return (X_tr, Y_tr_scaled,
            X_va, Y_va_scaled,
            X_te, Y_te_raw_arr,
            feat_scaler, tgt_scalers,
            feature_cols)


def make_sequences(X: np.ndarray, Y: np.ndarray, lookback: int = LOOKBACK):
    """Sliding window → (N-lb, lb, n_features),  (N-lb, 4)."""
    Xs, Ys = [], []
    for i in range(lookback, len(X)):
        Xs.append(X[i - lookback: i])
        Ys.append(Y[i])
    return np.array(Xs, dtype=np.float32), np.array(Ys, dtype=np.float32)




class BahdanauAttention(layers.Layer):
    """
    Additive attention — identical to SOTA implementation.
        e_t = v^T · tanh(W_h · h_t)
        α_t = softmax(e)
        c   = Σ α_t · h_t
    """
    def __init__(self, units: int = 64, **kwargs):
        super().__init__(**kwargs)
        self.W = layers.Dense(units, use_bias=False)
        self.V = layers.Dense(1,     use_bias=False)

    def call(self, hidden_states):
        score   = self.V(tf.nn.tanh(self.W(hidden_states)))     # (B, T, 1)
        alpha   = tf.nn.softmax(score, axis=1)                   # (B, T, 1)
        context = tf.reduce_sum(alpha * hidden_states, axis=1)   # (B, 2*units)
        return context, tf.squeeze(alpha, -1)

    def get_config(self):
        return super().get_config()




def build_mtl_model(
    timesteps:    int,
    n_features:   int,
    lstm_units:   int   = LSTM_UNITS,
    dropout_rate: float = DROPOUT_RATE,
    dense_units:  int   = DENSE_UNITS,
    attn_units:   int   = ATTN_UNITS,
) -> Model:
    """
    MTL Architecture
    ────────────────
    SHARED ENCODER  (identical to SOTA single-task encoder)
      Input           →  (None, 24, n_features)
      BiLSTM₁ 128     →  (None, 24, 256)   return_sequences=True
      Dropout 0.4
      BiLSTM₂ 128     →  (None, 24, 256)   return_sequences=True
      Dropout 0.4
      BahdanauAttn    →  context (None, 256)

    TASK-SPECIFIC HEADS  (4 independent heads, one per pollutant)
      PM2.5 head:  Dense(64, ReLU) → Dense(1, linear)
      NO2   head:  Dense(64, ReLU) → Dense(1, linear)
      CO    head:  Dense(64, ReLU) → Dense(1, linear)
      Ozone head:  Dense(64, ReLU) → Dense(1, linear)

    All heads receive the SAME context vector from the shared encoder.
    Outputs concatenated → (None, 4).

    SOTA vs MTL comparison
    ──────────────────────
    SOTA : 4 × independent models, each trains its own full encoder
    MTL  : 1 shared encoder + 4 heads trained jointly
           → shared encoder learns cross-pollutant temporal patterns
           → roughly 3× fewer total parameters than 4 SOTA models
    """
    inp = keras.Input(shape=(timesteps, n_features), name="input")

    # ── Shared Encoder ────────────────────────────────────────────────────
    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True),
        name="shared_bilstm_1"
    )(inp)
    x = layers.Dropout(dropout_rate, name="shared_dropout_1")(x)

    x = layers.Bidirectional(
        layers.LSTM(lstm_units, return_sequences=True),
        name="shared_bilstm_2"
    )(x)
    x = layers.Dropout(dropout_rate, name="shared_dropout_2")(x)

    context, _ = BahdanauAttention(
        units=attn_units, name="shared_attention"
    )(x)

    # ── Task-Specific Heads ───────────────────────────────────────────────
    outputs = []
    for task_name in TARGET_KEYS:
        safe = task_name.replace(".", "_")
        h   = layers.Dense(
            dense_units, activation="relu",
            name=f"{safe}_dense"
        )(context)
        out = layers.Dense(
            1, activation="linear",
            name=f"{safe}_output"
        )(h)
        outputs.append(out)

    # Concatenate all task outputs → (None, 4)
    combined = layers.Concatenate(name="combined")(outputs)

    model = Model(inputs=inp, outputs=combined,
                  name="MTL_BiLSTM_Attention")


    model.compile(
        optimizer=keras.optimizers.RMSprop(learning_rate=LEARNING_RATE),
        loss="mse",
        metrics=["mae"],
    )
    return model



def train_model(model, X_tr, Y_tr, X_va, Y_va):
    callbacks = [
        keras.callbacks.EarlyStopping(
            monitor="val_loss", patience=PATIENCE,
            restore_best_weights=True, verbose=1
        ),
        keras.callbacks.ReduceLROnPlateau(
            monitor="val_loss", factor=0.5,
            patience=3, verbose=1
        ),
        keras.callbacks.ModelCheckpoint(
            filepath=os.path.join(OUTPUT_DIR, "best_mtl_model.keras"),
            monitor="val_loss", save_best_only=True, verbose=0
        ),
    ]

    history = model.fit(
        X_tr, Y_tr,
        validation_data=(X_va, Y_va),
        epochs=MAX_EPOCHS,
        batch_size=BATCH_SIZE,
        callbacks=callbacks,
        shuffle=False,      # preserve temporal order — same as SOTA
        verbose=1,
    )
    return history



def evaluate_mtl(model, X_te, Y_te_raw, tgt_scalers):
    """
    Predict all 4 pollutants simultaneously, inverse-transform each,
    compute RMSE / MAE / R² per pollutant.
    """
    Y_pred_scaled = model.predict(X_te, verbose=0)   # (N, 4)

    results = []
    print("\n" + "="*55)
    print("  MTL Test Results — Delhi")
    print("="*55)
    print(f"  {'Pollutant':<10} {'RMSE':>10} {'MAE':>10} {'R²':>10}")
    print("  " + "-"*42)

    for i, (display_name, col_name) in enumerate(TARGETS.items()):
        y_pred = tgt_scalers[col_name].inverse_transform(
            Y_pred_scaled[:, i].reshape(-1, 1)
        ).ravel()
        y_true = Y_te_raw[:, i]

        rmse = np.sqrt(mean_squared_error(y_true, y_pred))
        mae  = mean_absolute_error(y_true, y_pred)
        r2   = r2_score(y_true, y_pred)

        print(f"  {display_name:<10} {rmse:>10.4f} {mae:>10.4f} {r2:>10.4f}")
        results.append({
            "target": display_name,
            "RMSE": rmse, "MAE": mae, "R2": r2,
            "y_true": y_true, "y_pred": y_pred,
        })

    print("="*55)
    return results



def plot_training(history):
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    fig.suptitle("MTL Training — Stacked Bi-LSTM + Attention",
                 fontsize=12, fontweight="bold")

    axes[0].plot(history.history["loss"],     label="Train", color="#1e3a5f")
    axes[0].plot(history.history["val_loss"], label="Val",
                 color="#1e3a5f", linestyle="--", alpha=0.7)
    axes[0].set_title("Total Loss (MSE)")
    axes[0].set_xlabel("Epoch"); axes[0].set_ylabel("MSE")
    axes[0].legend(); axes[0].grid(alpha=0.3)

    axes[1].plot(history.history["mae"],     label="Train MAE", color="#16a34a")
    axes[1].plot(history.history["val_mae"], label="Val MAE",
                 color="#16a34a", linestyle="--", alpha=0.7)
    axes[1].set_title("MAE")
    axes[1].set_xlabel("Epoch"); axes[1].set_ylabel("MAE")
    axes[1].legend(); axes[1].grid(alpha=0.3)

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "mtl_training.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   Training plot → {path}")


def plot_predictions(results):
    for res in results:
        name  = res["target"]
        color = PALETTE[name]
        y_true, y_pred = res["y_true"], res["y_pred"]
        n = min(500, len(y_true))

        fig, axes = plt.subplots(1, 2, figsize=(14, 4))
        fig.suptitle(
            f"MTL — {name} (Delhi)\n"
            f"RMSE={res['RMSE']:.3f}  MAE={res['MAE']:.3f}  R²={res['R2']:.3f}",
            fontsize=12, fontweight="bold"
        )

        axes[0].plot(y_true[:n], label="Actual",    color="black", linewidth=0.8)
        axes[0].plot(y_pred[:n], label="Predicted", color=color,
                     linewidth=0.8, linestyle="--", alpha=0.85)
        axes[0].set_title(f"Prediction vs Actual (first {n} steps)")
        axes[0].set_xlabel("Time Step"); axes[0].set_ylabel(f"{name}")
        axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

        axes[1].scatter(y_true, y_pred, alpha=0.15, s=6, color=color)
        lims = [min(y_true.min(), y_pred.min()),
                max(y_true.max(), y_pred.max())]
        axes[1].plot(lims, lims, "k--", linewidth=1, label="Perfect fit")
        axes[1].set_title("Scatter: Actual vs Predicted")
        axes[1].set_xlabel("Actual"); axes[1].set_ylabel("Predicted")
        axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

        plt.tight_layout()
        path = os.path.join(OUTPUT_DIR, f"mtl_{name.lower()}.png")
        plt.savefig(path, dpi=150, bbox_inches="tight")
        plt.close()
        print(f"   Plot → {path}")


def plot_comparison(mtl_results):
    """Side-by-side SOTA vs MTL. Runs only if sota_metrics.csv exists."""
    sota_csv = os.path.join("sota_outputs", "sota_metrics.csv")
    if not os.path.exists(sota_csv):
        print("   sota_metrics.csv not found — skipping comparison plot.")
        return

    sota_df = pd.read_csv(sota_csv)
    mtl_df  = pd.DataFrame([
        {"target": r["target"], "RMSE": r["RMSE"],
         "MAE": r["MAE"], "R2": r["R2"]}
        for r in mtl_results
    ])

    metrics = ["RMSE", "MAE", "R2"]
    titles  = ["RMSE  (↓ better)", "MAE  (↓ better)", "R²  (↑ better)"]
    x, width = np.arange(len(TARGET_KEYS)), 0.35

    fig, axes = plt.subplots(1, 3, figsize=(18, 5))
    fig.suptitle("SOTA (Single-Task) vs MTL — Delhi Air Quality",
                 fontsize=13, fontweight="bold")

    for ax, metric, title in zip(axes, metrics, titles):
        sota_vals = [
            float(sota_df.loc[sota_df["target"] == t, metric].values[0])
            for t in TARGET_KEYS
        ]
        mtl_vals = [
            float(mtl_df.loc[mtl_df["target"] == t, metric].values[0])
            for t in TARGET_KEYS
        ]
        all_vals = sota_vals + mtl_vals

        b1 = ax.bar(x - width/2, sota_vals, width,
                    label="SOTA (Single-Task)", color="#94a3b8", edgecolor="white")
        b2 = ax.bar(x + width/2, mtl_vals,   width,
                    label="MTL (Ours)",          color="#2563eb", edgecolor="white")

        ax.set_title(title); ax.set_xticks(x)
        ax.set_xticklabels(TARGET_KEYS); ax.legend(fontsize=8)
        ax.grid(axis="y", alpha=0.3)

        for bar, v in zip(list(b1) + list(b2), all_vals):
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + max(all_vals) * 0.01,
                    f"{v:.3f}", ha="center", fontsize=7, fontweight="bold")

    plt.tight_layout()
    path = os.path.join(OUTPUT_DIR, "sota_vs_mtl_comparison.png")
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.close()
    print(f"   Comparison plot → {path}")



def main():
    print("─"*60)
    print("  MTL: Stacked Bi-LSTM + Attention — Delhi")
    print("  Tasks : PM2.5 | NO2 | CO | Ozone")
    print("  Loss  : Equal MSE per head (summed)")
    print("─"*60)

  
    print("\n[1/5] Loading data …")
    train_df = load_dataframe(TRAIN_PATH)
    val_df   = load_dataframe(VAL_PATH)
    test_df  = load_dataframe(TEST_PATH)
    print(f"   Train: {len(train_df)}  Val: {len(val_df)}  Test: {len(test_df)}")


    print("\n[2/5] Preprocessing …")
    (X_tr, Y_tr,
     X_va, Y_va,
     X_te, Y_te_raw,
     feat_scaler, tgt_scalers,
     feature_cols) = build_arrays(train_df, val_df, test_df)

    X_tr_seq, Y_tr_seq = make_sequences(X_tr, Y_tr)
    X_va_seq, Y_va_seq = make_sequences(X_va, Y_va)
    X_te_seq, Y_te_seq = make_sequences(X_te, Y_te_raw)

    print(f"   Input features ({len(feature_cols)}): {feature_cols}")
    print(f"   X_train: {X_tr_seq.shape}   Y_train: {Y_tr_seq.shape}")
    print(f"   X_val  : {X_va_seq.shape}   Y_val  : {Y_va_seq.shape}")
    print(f"   X_test : {X_te_seq.shape}   Y_test : {Y_te_seq.shape}")


    print("\n[3/5] Building model …")
    model = build_mtl_model(
        timesteps=LOOKBACK,
        n_features=X_tr_seq.shape[2],
    )
    model.summary()
    print(f"\n   Total parameters: {model.count_params():,}")

 
    print("\n[4/5] Training …")
    history = train_model(model, X_tr_seq, Y_tr_seq,
                          X_va_seq, Y_va_seq)


    print("\n[5/5] Evaluating …")
    mtl_results = evaluate_mtl(model, X_te_seq, Y_te_seq, tgt_scalers)


    metrics_df = pd.DataFrame([
        {"model":  "MTL_BiLSTM_Attention",
         "target": r["target"],
         "RMSE":   round(r["RMSE"], 4),
         "MAE":    round(r["MAE"],  4),
         "R2":     round(r["R2"],   4)}
        for r in mtl_results
    ])
    csv_path = os.path.join(OUTPUT_DIR, "mtl_metrics.csv")
    metrics_df.to_csv(csv_path, index=False)
    print(f"\n   Metrics saved → {csv_path}")


    print("\n   Generating plots …")
    plot_training(history)
    plot_predictions(mtl_results)
    plot_comparison(mtl_results)

    print("\nDone ✓")
    return model, mtl_results, metrics_df


if __name__ == "__main__":
    model, mtl_results, metrics_df = main()

────────────────────────────────────────────────────────────
  MTL: Stacked Bi-LSTM + Attention — Delhi
  Tasks : PM2.5 | NO2 | CO | Ozone
  Loss  : Equal MSE per head (summed)
────────────────────────────────────────────────────────────

[1/5] Loading data …
   Train: 52584  Val: 1416  Test: 744

[2/5] Preprocessing …
   Input features (18): ['pm10', 'no', 'nh3', 'nox', 'so2', 'bp', 'wind_speed', 'air_temp', 'humidity', 'rainfall', 'hour_sin', 'hour_cos', 'dow_sin', 'dow_cos', 'pm25', 'no2', 'co', 'ozone']
   X_train: (52560, 24, 18)   Y_train: (52560, 4)
   X_val  : (1392, 24, 18)   Y_val  : (1392, 4)
   X_test : (720, 24, 18)   Y_test : (720, 4)

[3/5] Building model …


Model: "MTL_BiLSTM_Attention"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input (InputLayer)  │ (None, 24, 18)    │          0 │ -                 │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bilstm_1     │ (None, 24, 256)   │    150,528 │ input[0][0]       │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout_1    │ (None, 24, 256)   │          0 │ shared_bilstm_1[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_bilstm_2     │ (None, 24, 256)   │    394,240 │ shared_dropout_1… │
│ (Bidirectional)     │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_dropout_2    │ (None, 24, 256)   │          0 │ shared_bilstm_2[… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ shared_attention    │ [(None, 256),     │     16,448 │ shared_dropout_2… │
│ (BahdanauAttention) │ (None, 24)]       │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ PM2_5_dense (Dense) │ (None, 64)        │     16,448 │ shared_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ NO2_dense (Dense)   │ (None, 64)        │     16,448 │ shared_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CO_dense (Dense)    │ (None, 64)        │     16,448 │ shared_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Ozone_dense (Dense) │ (None, 64)        │     16,448 │ shared_attention… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ PM2_5_output        │ (None, 1)         │         65 │ PM2_5_dense[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ NO2_output (Dense)  │ (None, 1)         │         65 │ NO2_dense[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ CO_output (Dense)   │ (None, 1)         │         65 │ CO_dense[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ Ozone_output        │ (None, 1)         │         65 │ Ozone_dense[0][0] │
│ (Dense)             │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ combined            │ (None, 4)         │          0 │ PM2_5_output[0][… │
│ (Concatenate)       │                   │            │ NO2_output[0][0], │
│                     │                   │            │ CO_output[0][0],  │
│                     │                   │            │ Ozone_output[0][… │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 627,268 (2.39 MB)

 Trainable params: 627,268 (2.39 MB)

 Non-trainable params: 0 (0.00 B)


   Total parameters: 627,268

[4/5] Training …
Epoch 1/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 18s 15ms/step - loss: 0.1736 - mae: 0.2423 - val_loss: 0.0389 - val_mae: 0.1462 - learning_rate: 0.0010
Epoch 2/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 12s 15ms/step - loss: 0.0882 - mae: 0.1702 - val_loss: 0.0263 - val_mae: 0.1184 - learning_rate: 0.0010
Epoch 3/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0779 - mae: 0.1568 - val_loss: 0.0228 - val_mae: 0.1085 - learning_rate: 0.0010
Epoch 4/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0729 - mae: 0.1490 - val_loss: 0.0227 - val_mae: 0.1084 - learning_rate: 0.0010
Epoch 5/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 10s 12ms/step - loss: 0.0697 - mae: 0.1448 - val_loss: 0.0209 - val_mae: 0.1041 - learning_rate: 0.0010
Epoch 6/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0671 - mae: 0.1405 - val_loss: 0.0221 - val_mae: 0.1082 - learning_rate: 0.0010
Epoch 7/50
822/822 ━━━━━━━━━━━━━━━━━━━━ 11s 13ms/step - loss: 0.0650 - mae: 0.1376 - val